# BEB-La-DII: Phase 1 Training (Awakening)
Этот блокнот оптимизирован для работы с зеркальным датасетом ресурсов. Он включает в себя:
1. Автоматическую настройку зеркальной структуры путей.
2. Инсталляцию необходимых зависимостей (включая bitsandbytes для учителя).
3. Полноценный цикл дистилляции с логированием в WandB.

In [ ]:
# 0. ПОЛУЧЕНИЕ И ОБНОВЛЕНИЕ ИСХОДНОГО КОДА
import os, sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/Laeryid/BEBLaDII"
REPO_NAME = "BEBLaDII"

if not os.path.exists(REPO_NAME):
    print(f"Клонирование репозитория {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Репозиторий {REPO_NAME} уже присутствует. Проверка обновлений...")
    !cd {REPO_NAME} && git pull

if os.path.exists(REPO_NAME) and REPO_NAME not in os.getcwd():
    os.chdir(REPO_NAME)
    print(f"Рабочая директория: {os.getcwd()}")

In [ ]:
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ И ПУТЕЙ
import subprocess
def install_packages():
    packages = ["transformers==4.57.2", "indexed-parquet-dataset", "optimum-intel[openvino]", "wandb", "accelerate", "bitsandbytes"]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_packages()

root_str = os.getcwd()
if root_str not in sys.path: sys.path.insert(0, root_str)
print(f"Корень проекта: {root_str}")

In [ ]:
import os, torch, json, wandb
from torch.optim import AdamW
from tqdm.auto import tqdm

try:
    from src.beb_la_dii.model.assembler import ModelAssembler
    from src.beb_la_dii.utils.loss import DistillationLoss
    from src.beb_la_dii.utils.data import get_dataloader
    from src.beb_la_dii.utils.experiment_tracker import ExperimentTracker
    from src.beb_la_dii.utils.tokenizer import get_tokenizer
    print("Модули проекта успешно импортированы.")
except ImportError as e: print(f"Ошибка импорта: {e}")

In [ ]:
# КОНФИГУРАЦИЯ
KAGGLE_INPUT_PATH = "/kaggle/input/datasets/bogdanbuliakov/bebladii-resources"
TEACHER_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
STAGE = 'awakening' 
LEARNING_RATE = 5e-5
EPOCHS = 1
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
def setup_mirrored_kaggle():
    """Настройка зеркальной структуры симлинков"""
    if not os.path.exists("/kaggle/input"): return
    print("--- Зеркальная настройка ресурсов Kaggle ---")
    if not os.path.exists(KAGGLE_INPUT_PATH):
        print(f"WARN: Датасет не найден по пути {KAGGLE_INPUT_PATH}")
        return
    mappings = [("components", "storage/components"), ("prebuilt", "storage/prebuilt"), ("data", "data")]
    for src_name, dst_path in mappings:
        src = os.path.join(KAGGLE_INPUT_PATH, src_name)
        dst = os.path.abspath(dst_path)
        if os.path.exists(src):
            if os.path.exists(dst):
                if os.path.islink(dst): os.unlink(dst)
                else: shutil.rmtree(dst)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            os.symlink(src, dst)
            print(f"УСПЕХ: {dst_path} -> {src_name}")
setup_mirrored_kaggle()

In [ ]:
# СБОРКА МОДЕЛИ И НАСТРОЙКА ГРАДИЕНТОВ
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Инициализация на {device}...")

assembler = ModelAssembler()
distiller = assembler.assemble_phase1_distiller(teacher_id=TEACHER_NAME, device_map="auto")

# 1. Замораживаем всё
for p in distiller.parameters(): p.requires_grad = False

# 2. Размораживаем студента и проекторы
for p in distiller.student.parameters(): p.requires_grad = True
for p in distiller.input_projector.parameters(): p.requires_grad = True
for proj in distiller.feature_projectors.values():
    for p in proj.parameters(): p.requires_grad = True

print(f"Обучаемых параметров: {sum(p.numel() for p in distiller.parameters() if p.requires_grad):,}")

In [ ]:
# ПОДГОТОВКА ОБУЧЕНИЯ
tokenizer = get_tokenizer()
dataloader = get_dataloader(stage=STAGE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
optimizer = AdamW(filter(lambda p: p.requires_grad, distiller.parameters()), lr=LEARNING_RATE)
criterion = DistillationLoss()
tracker = ExperimentTracker(project_root=".", stage=STAGE)

# Инициализация WandB (если есть ключ)
if os.environ.get("WANDB_API_KEY"):
    wandb.init(project="BEBLaDII", name=f"phase1-{STAGE}")
else:
    print("WANDB_API_KEY не найден. Логирование только локальное.")

In [ ]:
# ЦИКЛ ОБУЧЕНИЯ
distiller.train()
print("--- Запуск обучения Фазы 1 ---")

for epoch in range(EPOCHS):
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}")
    accum_loss = 0.0
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        
        # Прямой проход
        #outputs, targets = distiller(input_ids, mask) 
        # Внимание: Убедитесь, что сигнатура distiller совпадает с возвращаемыми значениями
        try:
             student_states, teacher_targets = distiller(input_ids, mask)
             loss = criterion(student_states, teacher_targets) / GRAD_ACCUM_STEPS
             loss.backward()
             accum_loss += loss.item()
        except Exception as e:
             print(f"Ошибка на шаге {step}: {e}")
             continue
        
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(distiller.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
            avg_loss = accum_loss * GRAD_ACCUM_STEPS
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            
            # Логирование
            tracker.log_step(step, {"loss": avg_loss})
            if wandb.run: wandb.log({"loss": avg_loss, "step": step})
            accum_loss = 0.0
            
    # Сохранение эпохи
    tracker.save_checkpoint(distiller.state_dict(), name=f"phase1_epoch_{epoch}")

print("Обучение завершено!")